# Advanced Feature Engineering for Demand Forecasting

This notebook applies advanced feature engineering techniques to a real-world sales dataset. It introduces a custom promotion detection algorithm and leverages the resulting feature to improve demand forecasting.

**Techniques Covered:**
- **Promotion Detection:** Identifying sales spikes that indicate promotions.
- **Multi-Level Temporal Encodings:** Capturing daily, weekly, and monthly patterns.
- **Price Sensitivity:** Modeling the impact of unit price changes.
- **Promotion Lag Effects:** Accounting for the delayed impact of promotions.
- **Automated Feature Importance:** Using SHAP values to understand feature contributions.

## 1. Setup and Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from typing import Union, Tuple, List

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
shap.initjs()

## 2. Data Loading and Promotion Detection

In [ ]:
def detect_promotions(df: pd.DataFrame, explainability: bool = False) -> Union[pd.DataFrame, Tuple[pd.DataFrame, List[str]]]:
    """\n    Detects promotions in sales data based on sharp increases in Total_units.\n\n    Args:\n        df: DataFrame with sales data.\n        explainability: If True, returns a tuple with the DataFrame and a list of explanations.\n\n    Returns:\n        A DataFrame with 'is_promotion' column, or a tuple with the DataFrame and explanations.\n    """\n    df['Proc_date'] = pd.to_datetime(df['Proc_date'])\n    df = df.sort_values(by=['store_id', 'item', 'Proc_date'])\n    df['units_diff'] = df.groupby(['store_id', 'item'])['Total_units'].diff().fillna(0)\n    stats = df.groupby(['store_id', 'item'])['units_diff'].agg(['mean', 'std']).reset_index()\n    df = pd.merge(df, stats, on=['store_id', 'item'], suffixes=('', '_stats'))\n    df['std_stats'] = df['std_stats'].fillna(0)\n    df['threshold'] = df['mean_stats'] + (2 * df['std_stats'])\n    df['is_promotion'] = (df['units_diff'] > 0) & (df['units_diff'] > df['threshold'])\n    
    explanations = []
    if explainability:
        promotion_rows = df[df['is_promotion']].copy()
        for _, row in promotion_rows.iterrows():
            explanation = (
                f"Promotion detected for item {row['item']} at store {row['store_id']} on {row['Proc_date'].date()}:\n"
                f"  - Sales jumped by {row['units_diff']:.2f} units.\n"
                f"  - This exceeded the threshold of {row['threshold']:.2f}.\n"
                f"  - (Calculated from mean diff: {row['mean_stats']:.2f}, std diff: {row['std_stats']:.2f})"
            )
            explanations.append(explanation)
            
    df = df.drop(columns=['units_diff', 'mean_stats', 'std_stats', 'threshold'])
    
    if explainability:
        return df, explanations
    else:
        return df

# Load and process data
sales_df = pd.read_csv('sales.csv')
df, explanations = detect_promotions(sales_df.copy(), explainability=True)

# Create a 'unit_price' column
df['unit_price'] = df['Total_Retail_] / df['Total_units']
df['unit_price'] = df['unit_price'].fillna(0)

# Set Proc_date as index
df = df.set_index('Proc_date')

print("--- Promotion Explanations ---")
if explanations:
    for exp in explanations:
        print(exp)
else:
    print("No promotions were detected.")

## 3. Feature Engineering

In [ ]:
def feature_engineer(df):
    df_eng = df.copy()
    
    # 1. Multi-Level Temporal Encodings
    df_eng['hour'] = df_eng.index.hour
    df_eng['day_of_week'] = df_eng.index.dayofweek
    df_eng['day_of_year'] = df_eng.index.dayofyear
    df_eng['month'] = df_eng.index.month
    df_eng['week_of_year'] = df_eng.index.isocalendar().week.astype(int)
    
    # Cyclical features for hour and month
    df_eng['hour_sin'] = np.sin(2 * np.pi * df_eng['hour'] / 24)
    df_eng['hour_cos'] = np.cos(2 * np.pi * df_eng['hour'] / 24)
    df_eng['month_sin'] = np.sin(2 * np.pi * df_eng['month'] / 12)
    df_eng['month_cos'] = np.cos(2 * np.pi * df_eng['month'] / 12)
    
    # 2. Price Sensitivity Indicator
    # Use a global mean for rolling average due to sparse data
    df_eng['price_vs_avg'] = df_eng['unit_price'] - df_eng['unit_price'].mean()
    
    # 3. Promotion Lag Effects
    df_eng['promo_lag_1'] = df_eng['is_promotion'].shift(1).fillna(0)
    df_eng['promo_lag_2'] = df_eng['is_promotion'].shift(2).fillna(0)
    df_eng['promo_rolling_24h'] = df_eng['is_promotion'].rolling(24, min_periods=1).sum().fillna(0)
    
    # 4. Stock-Out Impact Modeling (Simplified)
    df_eng['rolling_avg_sales'] = df_eng['Total_units'].rolling(3, min_periods=1).mean().fillna(0)
    high_sales_threshold = df_eng['rolling_avg_sales'].quantile(0.90)
    df_eng['stock_out_risk'] = (df_eng['rolling_avg_sales'] > high_sales_threshold).astype(int)
    df_eng['stock_out_impact'] = df_eng['stock_out_risk'].shift(1).fillna(0)
    
    # Drop intermediate and original columns
    df_eng = df_eng.drop(['rolling_avg_sales', 'stock_out_risk', 'Item_Description', 'Size'], axis=1)
    
    return df_eng.dropna()

df_engineered = feature_engineer(df)
print("\nEngineered features:")
print(df_engineered.head())

## 4. Model Training and Evaluation

In [ ]:
# Define feature sets
TARGET = 'Total_units'
NON_FEATURES = [TARGET, 'store_id', 'item', 'Total_Retail_, 'Total_Cost_, 'Total_lbs']

BASELINE_FEATURES = ['unit_price', 'is_promotion', 'hour', 'day_of_week', 'month']
ADVANCED_FEATURES = [col for col in df_engineered.columns if col not in NON_FEATURES and col not in ['is_promotion']]

# Ensure all baseline features are present
for col in BASELINE_FEATURES:
    if col not in df_engineered.columns:
        raise ValueError(f"Column '{col}' not found in engineered dataframe")

# Split data
X = df_engineered.drop(NON_FEATURES, axis=1, errors='ignore')
y = df_engineered[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train Baseline Model
model_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_base.fit(X_train[BASELINE_FEATURES], y_train)
preds_base = model_base.predict(X_test[BASELINE_FEATURES])

# Train Advanced Model
model_adv = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_adv.fit(X_train[ADVANCED_FEATURES], y_train)
preds_adv = model_adv.predict(X_test[ADVANCED_FEATURES])

# Evaluate
rmse_base = np.sqrt(mean_squared_error(y_test, preds_base))
mae_base = mean_absolute_error(y_test, preds_base)
rmse_adv = np.sqrt(mean_squared_error(y_test, preds_adv))
mae_adv = mean_absolute_error(y_test, preds_adv)

print("--- Baseline Model Performance ---")
print(f"RMSE: {rmse_base:.2f}")
print(f"MAE: {mae_base:.2f}")

print("--- Advanced Model Performance ---")
print(f"RMSE: {rmse_adv:.2f}")
print(f"MAE: {mae_adv:.2f}")

## 5. Feature Importance Analysis with SHAP

In [ ]:
# Explain the model's predictions using SHAP
explainer = shap.TreeExplainer(model_adv)
shap_values = explainer.shap_values(X_test[ADVANCED_FEATURES])

# Summary plot
shap.summary_plot(shap_values, X_test[ADVANCED_FEATURES], plot_type="bar")

In [ ]:
# More detailed summary plot
shap.summary_plot(shap_values, X_test[ADVANCED_FEATURES])

### Interpreting the SHAP Plots

The plots above show the contribution of each feature to the model's output. 

- The **bar chart** shows the average absolute SHAP value for each feature, indicating its overall importance.
- The **dot plot** (or summary plot) shows the SHAP values for each sample, providing more detail on the direction and magnitude of the feature's effect. For example, you can see how high or low values of a feature impact the prediction.